# Download Data

[Daily and Sports Activities](https://archive.ics.uci.edu/dataset/256/daily+and+sports+activities)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# %cd drive/My Drive/MIPT/Unsupervised/

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

# # Configuration - update this to your actual data path
# DATA_DIR = './data'  # The root directory containing a01, a02, etc.
# ACTIVITIES = [f'a{i:02d}' for i in range(1, 20)]  # a01 to a19
# SUBJECTS = [f'p{i}' for i in range(1, 9)]  # p1 to p8
# SEGMENTS = [f's{i:02d}' for i in range(1, 61)]  # s01 to s60

# # Sensor column names (45 total)
# UNITS = ['T', 'RA', 'LA', 'RL', 'LL']  # Torso, Right Arm, Left Arm, Right Leg, Left Leg
# SENSORS = ['xacc', 'yacc', 'zacc', 'xgyro', 'ygyro', 'zgyro', 'xmag', 'ymag', 'zmag']
# COLUMNS = [f"{unit}_{sensor}" for unit in UNITS for sensor in SENSORS]

# # Activity label mapping
# ACTIVITY_LABELS = {
#     1: 'sitting', 2: 'standing', 3: 'lying on back', 4: 'lying on right side',
#     5: 'ascending stairs', 6: 'descending stairs', 7: 'standing in elevator',
#     8: 'moving in elevator', 9: 'walking in parking lot', 10: 'walking on treadmill (flat)',
#     11: 'walking on treadmill (inclined)', 12: 'running on treadmill',
#     13: 'exercising on stepper', 14: 'exercising on cross trainer',
#     15: 'cycling horizontal', 16: 'cycling vertical', 17: 'rowing',
#     18: 'jumping', 19: 'playing basketball'
# }

# def read_segment_file(file_path):
#     """Read a single segment file and return as numpy array"""
#     try:
#         return np.loadtxt(file_path, delimiter=',')
#     except Exception as e:
#         print(f"Error reading {file_path}: {str(e)}")
#         return None

# def build_full_dataframe():
#     """Constructs a DataFrame with all 125 timepoints per segment"""
#     records = []

#     for activity in tqdm(ACTIVITIES, desc='Processing Activities'):
#         activity_num = int(activity[1:])
#         activity_name = ACTIVITY_LABELS[activity_num]

#         for subject in tqdm(SUBJECTS, desc=f'Processing {activity}', leave=False):
#             subject_num = int(subject[1:])

#             for segment in SEGMENTS:
#                 segment_num = int(segment[1:])
#                 file_path = os.path.join(DATA_DIR, activity, subject, f"{segment}.txt")

#                 segment_data = read_segment_file(file_path)
#                 if segment_data is None:
#                     continue

#                 # Verify data shape (should be 125 timepoints × 45 sensors)
#                 if segment_data.shape != (125, 45):
#                     print(f"Unexpected shape {segment_data.shape} in {file_path}")
#                     continue

#                 # Create a record for each timepoint
#                 for timepoint in range(125):
#                     record = {
#                         'activity_num': activity_num,
#                         'activity_name': activity_name,
#                         'subject': subject_num,
#                         'segment': segment_num,
#                         'timepoint': timepoint,
#                         'timestamp_ms': timepoint * 40  # 25Hz = 40ms per sample
#                     }
#                     # Add all sensor readings
#                     record.update(zip(COLUMNS, segment_data[timepoint]))
#                     records.append(record)

#     return pd.DataFrame(records)

# # Build the complete DataFrame
# print("Building DataFrame...")
# df = build_full_dataframe()

# # Print summary
# print("\nDataFrame created successfully!")
# print(f"Total segments processed: {len(df) // 125}")
# print(f"DataFrame shape: {df.shape}")
# print("\nSample data:")
# print(df.iloc[1000:1005][['activity_name', 'subject', 'segment', 'timepoint'] + COLUMNS[:3]])

In [ ]:
# # Save to efficient format
# df.to_parquet('full_sensor_data.parquet', index=False)

In [ ]:
# pd.set_option('display.max_columns', None)

In [ ]:
# display(df.head(3))

# Different Dimensionality Reduction + Clustering Algorithms

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import silhouette_score, adjusted_rand_score
from tqdm import tqdm

In [ ]:
df = pd.read_parquet('full_sensor_data.parquet')

In [ ]:
def apply_clustering(features, n_clusters=19):
    results = {}

    # K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    results['kmeans'] = kmeans.fit_predict(features)

    # DBSCAN (auto-tune eps)
    from sklearn.neighbors import NearestNeighbors
    neigh = NearestNeighbors(n_neighbors=5)
    nbrs = neigh.fit(features)
    distances, _ = nbrs.kneighbors(features)
    eps = np.percentile(distances[:, 1], 95)  # More robust heuristic
    dbscan = DBSCAN(eps=eps, min_samples=5)
    results['dbscan'] = dbscan.fit_predict(features)

    # Hierarchical
    hierarchical = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    results['hierarchical'] = hierarchical.fit_predict(features)

    return results

In [ ]:
def extract_manual_features(df):
    """Calculate mean, std, and quartiles (25th, 75th) for each sensor in each segment"""
    sensor_columns = [col for col in df.columns if col.startswith(('T_', 'RA_', 'LA_', 'RL_', 'LL_'))]

    features = []
    meta = []

    # Group by segment
    grouped = df.groupby(['activity_num', 'subject', 'segment'])

    for (activity, subject, segment), group in tqdm(grouped, desc="Extracting features"):
        # Calculate statistics
        segment_data = group[sensor_columns]
        means = segment_data.mean(axis=0)
        stds = segment_data.std(axis=0)
        q25 = segment_data.quantile(0.25, axis=0)
        q75 = segment_data.quantile(0.75, axis=0)

        # Create feature vector (4 stats per sensor)
        feature_vector = np.zeros(180)  # 45 sensors * 4 stats
        for i in range(45):
            feature_vector[4*i] = means.iloc[i]     # mean
            feature_vector[4*i + 1] = stds.iloc[i]  # std
            feature_vector[4*i + 2] = q25.iloc[i]   # 25th percentile
            feature_vector[4*i + 3] = q75.iloc[i]   # 75th percentile

        features.append(feature_vector)
        meta.append({
            'activity': activity,
            'subject': subject,
            'segment': segment
        })

    features = np.array(features)
    meta_df = pd.DataFrame(meta)
    return features, meta_df

def apply_clustering(features, n_clusters=19):
    results = {}

    # K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    results['kmeans'] = kmeans.fit_predict(features)

    # DBSCAN (auto-tune eps)
    from sklearn.neighbors import NearestNeighbors
    neigh = NearestNeighbors(n_neighbors=5)
    nbrs = neigh.fit(features)
    distances, _ = nbrs.kneighbors(features)
    eps = np.percentile(distances[:, 1], 95)  # More robust heuristic
    dbscan = DBSCAN(eps=eps, min_samples=5)
    results['dbscan'] = dbscan.fit_predict(features)

    # Hierarchical
    hierarchical = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    results['hierarchical'] = hierarchical.fit_predict(features)

    return results

# Dimensionality Reduction and Evaluation
def reduce_and_cluster(features, meta_df, n_clusters=19):
    results = []

    # Scale features
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    # Define reduction methods
    reducers = {
        'PCA': PCA(n_components=2),
        't-SNE': TSNE(n_components=2, perplexity=30),
        'UMAP': umap.UMAP(n_components=2, n_neighbors=10)
    }

    # For each reduction method
    for name, reducer in reducers.items():
        print(f"\nProcessing {name}...")

        # Reduce dimensions
        reduced_data = reducer.fit_transform(scaled_features)

        # Apply clustering
        clustering_results = apply_clustering(reduced_data, n_clusters)

        # Evaluate
        print(f"Clustering Evaluation for {name}:")
        print("{:<15} {:<15} {:<15}".format('Method', 'Silhouette', 'ARI'))

        true_labels = meta_df['activity'] - 1  # Make 0-based

        for cluster_name, labels in clustering_results.items():
            if len(np.unique(labels)) > 1:
                sil = silhouette_score(reduced_data, labels)
                ari = adjusted_rand_score(true_labels, labels)
                print("{:<15} {:<15.3f} {:<15.3f}".format(cluster_name, sil, ari))
            else:
                print(f"{cluster_name}: Only 1 cluster found")

        # Store results for visualization
        results.append({
            'name': name,
            'data': reduced_data,
            'clusterings': clustering_results
        })

    return results

# Visualization
def plot_reduction_results(results, meta_df):
    plt.figure(figsize=(18, 12))

    for i, result in enumerate(results, 1):
        for j, (cluster_name, cluster_labels) in enumerate(result['clusterings'].items(), 1):
            plt.subplot(3, 3, (i-1)*3 + j)

            # Plot by cluster
            scatter = plt.scatter(result['data'][:,0], result['data'][:,1],
                                c=cluster_labels, cmap='tab20', alpha=0.6)

            # Add activity centroids
            for act in meta_df['activity'].unique():
                mask = meta_df['activity'] == act
                plt.scatter(np.median(result['data'][mask,0]), np.median(result['data'][mask,1]),
                          marker='x', s=100, linewidths=2, color='black')

            plt.title(f"{result['name']} + {cluster_name}")
            plt.xlabel('Component 1')
            plt.ylabel('Component 2')

            if cluster_name == 'kmeans':
                plt.colorbar(scatter, label='Cluster')

    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Extract features
print("Extracting manual features...")
features, meta_df = extract_manual_features(df)

In [ ]:
# 2. Reduce dimensions and cluster
print("\nReducing dimensions and clustering...")
results = reduce_and_cluster(features, meta_df)

# 3. Visualize
print("\nGenerating visualizations...")
plot_reduction_results(results, meta_df)